# Unix 3B: local mapping, validation, and coverage

This lesson runs the supplied Nanopore mapping workflow locally and interprets its QA outputs.

## Learning outcomes

By the end you should be able to produce a sorted and indexed BAM, validate it, report mapping statistics, and calculate depth with an explicit denominator.


## From file checks to biological interpretation

Unix 3A established that the inputs are present and internally consistent. Unix 3B runs the mapping workflow and then asks whether the result is trustworthy enough to interpret.

The practical question is simple: how well do the reads from `barcode09/` cover the supplied *Haloferax volcanii* reference? The computational answer requires several linked steps: map reads, sort and index alignments, validate the BAM, calculate depth, and summarise coverage.


## 1. Prerequisites and output location

Start Jupyter in `UNIX_session_3`. Complete Unix 3A first, or review its file-product diagram before running this notebook.

This notebook writes only to `work/notebook_run/`; rerunning it replaces the named demonstration outputs there. The raw reads and reference should remain unchanged.


In [1]:
%%bash
./preflight.sh


reference_records reference_bases: 5 4012900
read_records read_bases: 16801 60751725
fastq_files: 5
minimap2: 2.28-r1209
samtools: samtools 1.21
Preflight passed.


## 2. Scientific workflow

Minimap2 is a fast sequence aligner. It compares sequencing reads with a reference and reports where each read aligns. For Oxford Nanopore genomic reads we use the `map-ont` preset:

```bash
minimap2 -ax map-ont reference/HVol_Complete.fasta barcode09/*.fastq.gz
```

The key options are:

- `-a`: write SAM alignment output;
- `-x map-ont`: use settings designed for Oxford Nanopore genomic reads;
- `-t 2`: use two worker threads.

SAM output can be very large, so the workflow pipes it directly into `samtools sort`. Samtools then creates a coordinate-sorted BAM file, and `samtools index` creates the `.bai` index.

```text
minimap2 SAM stream -> samtools sort -> sorted BAM -> samtools index -> BAI
```

Run this locally so you can see the files being produced and checked as the workflow progresses.


### SAM, BAM, sorted BAM, and BAI

A **SAM** file is a text alignment file. It is useful for learning because you can inspect it with ordinary Unix tools, but it can become very large.

A **BAM** file stores the same kind of alignment information in a compressed binary format. Most downstream tools prefer BAM because it is smaller and faster to process.

A **sorted BAM** is ordered by reference coordinate. Sorting is required for many operations that summarise or index alignments by genomic position.

A **BAI** file is an index for a sorted BAM. It lets tools jump to regions of the reference without reading the whole alignment file from the beginning.


In [2]:
%%bash
set -euo pipefail
run_dir=work/notebook_run
mkdir -p "$run_dir"

minimap2 -t 2 -ax map-ont \
    reference/HVol_Complete.fasta barcode09/*.fastq.gz \
    2> "$run_dir/minimap2.log" \
  | samtools sort -@ 1 -o "$run_dir/aligned_barcode09.sorted.bam" -

samtools index "$run_dir/aligned_barcode09.sorted.bam"


## 3. Alignment validation

A result file existing on disk is not the same as a result being trustworthy. The checks below answer different questions:

- `samtools quickcheck` asks whether the BAM appears structurally intact.
- `test -s file.bai` asks whether the index file exists and is non-empty.
- `samtools flagstat` summarises alignment categories such as mapped, secondary, and supplementary records.

These checks do not prove that the correct sample and reference were selected. For that, retain the command, input identities, software versions, and logs.


In [3]:
%%bash
set -euo pipefail
bam=work/notebook_run/aligned_barcode09.sorted.bam
samtools quickcheck -v "$bam"
test -s "$bam.bai"
samtools flagstat "$bam" | head -n 8


17780 + 0 in total (QC-passed reads + QC-failed reads)
16801 + 0 primary
798 + 0 secondary
181 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
17398 + 0 mapped (97.85% : N/A)
16419 + 0 primary mapped (97.73% : N/A)


### Expected alignment checkpoint

The validated local run produced 16,801 primary records, 16,419 primary mapped records (97.73%), 798 secondary records, and 181 supplementary records. Record your Minimap2 and Samtools versions when comparing results.


### Primary, secondary, and supplementary alignments

Long-read mappers may report more than one alignment record for a read.

A **primary alignment** is the main placement chosen by the mapper. A **secondary alignment** is another plausible placement for the same read. A **supplementary alignment** represents an additional part of a split alignment, which can happen when a read spans a structural rearrangement, repeat, or chimeric region.

When reporting mapping totals, state whether your count includes only primary alignments or also includes secondary and supplementary records.


## 4. Complete-reference depth

Coverage depth is the number of aligned reads covering a reference position. If a base has depth 0, no read covers it. If it has depth 12, twelve alignments cover it.

`samtools depth` writes a tab-separated table:

```text
reference_name    position    depth
```

The denominator matters. By default, `samtools depth` can omit positions with zero depth. `-a` includes zero-depth positions on reference sequences represented in the BAM. `-aa` also includes reference sequences with no alignments at all.

For this practical we use `-aa` because the teaching question concerns the complete reference, not just the parts that received reads.


In [4]:
%%bash
set -euo pipefail
bam=work/notebook_run/aligned_barcode09.sorted.bam
depth=work/notebook_run/coverage_barcode09.tsv
samtools depth -aa "$bam" > "$depth"

awk '{sum += $3; if ($3 > 0) covered++; total++}
     END {printf "positions=%d covered=%d mean_depth=%.3f breadth=%.3f%%\n", total, covered, sum/total, 100*covered/total}' "$depth"


positions=4012900 covered=4005120 mean_depth=14.268 breadth=99.806%


### Expected coverage checkpoint

The validated run produced 4,012,900 rows, 4,005,120 covered positions, mean depth 14.268 across the complete reference, and breadth 99.806%. The wholly uncovered 6,359-base plasmid remains in the denominator. A large difference should trigger investigation before interpretation.


### Interpreting breadth and mean depth

Mean depth is the average number of aligned reads per reference position. Breadth is the percentage of reference positions with depth greater than zero.

A high mean depth can hide a gap if reads pile up unevenly. A high breadth can still have low confidence if most covered positions have very low depth. Good reporting therefore includes both mean depth and breadth, plus enough command detail for someone else to reproduce the denominator.


## 5. Per-reference summaries

The reference contains multiple records: a chromosome and plasmids. A single whole-reference mean can hide important differences between them.

Per-reference summaries help answer questions such as:

- Is the chromosome covered differently from the plasmids?
- Is one reference sequence entirely absent from the reads?
- Would the whole-reference mean change if an uncovered plasmid were excluded?

This is why biological interpretation should be connected back to the reference structure inspected in Unix 3A.


In [5]:
%%bash
awk '{sum[$1]+=$3; n[$1]++; if($3>0) covered[$1]++}
     END {for (ref in n) printf "%s\t%d\t%.3f\t%.3f%%\n", ref, n[ref], sum[ref]/n[ref], 100*covered[ref]/n[ref]}' \
    work/notebook_run/coverage_barcode09.tsv | sort


NC_013964.1	437906	12.442	100.000%
NC_013965.1	6359	0.000	0.000%
NC_013966.1	635786	17.285	99.863%
NC_013967.1	2847757	13.275	99.981%
NC_013968.1	85092	35.435	100.000%


## 6. Filtering decisions

Mapping summaries may count primary, secondary, and supplementary records differently. Depth can also change with mapping-quality filters, base-quality filters, overlap handling, and whether zero-depth positions are retained.

Avoid reporting a bare number such as "14x coverage" without the command that produced it. A stronger report says what was counted, what was excluded, and which reference length formed the denominator.


## 7. Local results to retain

Keep the evidence needed to reconstruct and defend the local analysis:

- the commands run in this notebook;
- software versions;
- sorted BAM and BAI when required;
- mapping and depth summaries;
- checksums for files that are copied or compared;
- notes on any filtering or denominator choices.

The streamed SAM is intentionally not retained because it is a large intermediate product.


## Next steps...

Before moving on, make sure you can answer these in your notes.

1. Why does the command use `-ax map-ont`?
2. What is the difference between SAM, BAM, sorted BAM, and BAI?
3. What does `samtools quickcheck` establish, and what does it not establish?
4. Why is `-aa` important for the reported complete-reference mean depth?
5. Why might chromosome-level and plasmid-level coverage tell a different story from the whole-reference mean?
6. Which evidence shows the local BAM and its index came from a reproducible workflow?
